# NB_02 — Electroplated Bi Process Window

**Engineering question**

> What electroplated-bismuth thickness and microstructure process window preserves Gaussian-like spectral response while maximizing x-ray stopping power?

This notebook uses **Engineering Objects**, not individual papers, as its primary inputs.


## Workflow

```text
Engineering Objects
    ↓
Absorber
Electroplating
TES
    ↓
Candidate process variables
    ↓
Evidence matrix
    ↓
Process window candidates
    ↓
Validation experiments
```

In [ ]:
from pathlib import Path
import yaml
import pandas as pd

def find_repo_root(start=None):
    start=(Path.cwd() if start is None else Path(start)).resolve()
    for c in (start,*start.parents):
        if (c/'engineering_navigator').exists():
            return c
    raise FileNotFoundError("Repository root not found")

ROOT=find_repo_root()
OBJ=ROOT/'engineering_navigator'/'engineering_objects'

def load(name):
    return yaml.safe_load((OBJ/f"{name}.yaml").read_text())

absorber=load("absorber")
electroplating=load("electroplating")
tes=load("tes")
print("Loaded engineering objects.")


## Process variables

In [ ]:
rows=[]
for v in electroplating.get("variables",[]):
    rows.append({
        "variable":v["id"],
        "unit":v.get("unit",""),
        "candidate_range":"",
        "status":"open"
    })
process_df=pd.DataFrame(rows)
process_df


## Coupled detector variables

In [ ]:
rows=[]
for v in absorber.get("variables",[]):
    rows.append({"object":"absorber","variable":v["id"],"unit":v.get("unit","")})
for v in tes.get("variables",[]):
    rows.append({"object":"tes","variable":v["id"],"unit":v.get("unit","")})
coupled_df=pd.DataFrame(rows)
coupled_df


## Candidate process window

In [ ]:
window=pd.DataFrame([
{
"engineering_object":"electroplating",
"optimization_target":"Tail-free spectral response",
"primary_variables":"current_density, plating_rate, Bi_thickness, grain_size",
"measured_outputs":"energy_resolution, low_energy_tail_fraction, quantum_efficiency",
"status":"requires_experiments"
}
])
window


## Proposed validation matrix

In [ ]:
validation=pd.DataFrame([
["Current density","Sweep","Grain size","SEM"],
["Bi thickness","Sweep","Low-energy tail","Spectrum"],
["Plating rate","Sweep","Energy resolution","TES"],
["Bath chemistry","Controlled","Repeatability","Multiple wafers"],
],columns=["Variable","Strategy","Response","Measurement"])
validation


## Outputs

Write in future versions:

- `candidate_process_window.csv`
- `validation_matrix.csv`
- `optimization_targets.json`

Version 1 establishes the object-driven engineering workflow. Subsequent versions will populate quantitative ranges from accumulated evidence.
